In [1]:
import polars as pl
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
import kagglehub

path = kagglehub.dataset_download("desolution01/messy-employee-dataset")

In [5]:
csv_file = [f for f in os.listdir(path) if f.endswith(".csv")][0]

In [7]:
df = pl.read_csv(os.path.join(path,csv_file))

In [10]:
df.columns

['Employee_ID',
 'First_Name',
 'Last_Name',
 'Age',
 'Department_Region',
 'Status',
 'Join_Date',
 'Salary',
 'Email',
 'Phone',
 'Performance_Score',
 'Remote_Work']

In [9]:
df.head(100)

Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
str,str,str,i64,str,str,str,str,str,i64,str,bool
"""EMP1000""","""Bob""","""Davis""",25,"""DevOps-California""","""Active""","""4/2/2021""","""59767.65""","""bob.davis@example.com""",-1651623197,"""Average""",true
"""EMP1001""","""Bob""","""Brown""",null,"""Finance-Texas""","""Active""","""7/10/2020""","""65304.66""","""bob.brown@example.com""",-1898471390,"""Excellent""",true
"""EMP1002""","""Alice""","""Jones""",null,"""Admin-Nevada""","""Pending""","""12/7/2023""","""88145.9""","""alice.jones@example.com""",-5596363211,"""Good""",true
"""EMP1003""","""Eva""","""Davis""",25,"""Admin-Nevada""","""Inactive""","""11/27/2021""","""69450.99""","""eva.davis@example.com""",-3476490784,"""Good""",true
"""EMP1004""","""Frank""","""Williams""",25,"""Cloud Tech-Florida""","""Active""","""1/5/2022""","""109324.61""","""frank.williams@example.com""",-1586734256,"""Poor""",false
…,…,…,…,…,…,…,…,…,…,…,…
"""EMP1095""","""Heidi""","""Miller""",25,"""Cloud Tech-California""","""Active""","""4/8/2022""","""98425.1""","""heidi.miller@example.com""",-4610339936,"""Average""",true
"""EMP1096""","""Frank""","""Johnson""",40,"""Sales-Illinois""","""Inactive""","""5/21/2022""","""119764.2""","""frank.johnson@example.com""",-8861405785,"""Average""",false
"""EMP1097""","""Frank""","""Brown""",40,"""HR-California""","""Inactive""","""4/17/2021""","""58445.91""","""frank.brown@example.com""",-1706304890,"""Excellent""",false


In [12]:
print(df.schema)
print(df.null_count())
print(df.glimpse())

Schema({'Employee_ID': String, 'First_Name': String, 'Last_Name': String, 'Age': Int64, 'Department_Region': String, 'Status': String, 'Join_Date': String, 'Salary': String, 'Email': String, 'Phone': Int64, 'Performance_Score': String, 'Remote_Work': Boolean})
shape: (1, 12)
┌─────────────┬────────────┬───────────┬─────┬───┬───────┬───────┬───────────────────┬─────────────┐
│ Employee_ID ┆ First_Name ┆ Last_Name ┆ Age ┆ … ┆ Email ┆ Phone ┆ Performance_Score ┆ Remote_Work │
│ ---         ┆ ---        ┆ ---       ┆ --- ┆   ┆ ---   ┆ ---   ┆ ---               ┆ ---         │
│ u32         ┆ u32        ┆ u32       ┆ u32 ┆   ┆ u32   ┆ u32   ┆ u32               ┆ u32         │
╞═════════════╪════════════╪═══════════╪═════╪═══╪═══════╪═══════╪═══════════════════╪═════════════╡
│ 0           ┆ 0          ┆ 0         ┆ 211 ┆ … ┆ 0     ┆ 0     ┆ 0                 ┆ 0           │
└─────────────┴────────────┴───────────┴─────┴───┴───────┴───────┴───────────────────┴─────────────┘
Rows: 1020
Column

In [14]:
def split_department_region(df: pl.DataFrame) -> pl.DataFrame:

    return (
        df.with_columns(
            pl.col("Department_Region").str.split("-").list.get(0).alias("Department"),
            pl.col("Department_Region").str.split("-").list.get(1).alias("Region"),
        )
        .drop("Department_Region")
    )


def parse_dates(df: pl.DataFrame, cols: list[str], fmt: str) -> pl.DataFrame:

    return df.with_columns([
        pl.col(c).str.to_date(fmt, strict=False).alias(c)
        for c in cols
    ])


def cast_numeric(df: pl.DataFrame, schema: dict[str, pl.DataType]) -> pl.DataFrame:


    return df.with_columns([
        pl.col(c).cast(t, strict=False)
        for c, t in schema.items()
    ])


def nullify_invalid(df: pl.DataFrame, col: str, condition) -> pl.DataFrame:

    return df.with_columns(
        pl.when(condition)
          .then(None)
          .otherwise(pl.col(col))
          .alias(col)
    )


def fill_nulls_median(df: pl.DataFrame, cols: list[str]) -> pl.DataFrame:

    return df.with_columns([
        pl.col(c).fill_null(pl.col(c).median().cast(pl.Int64))
        for c in cols
    ])


def cast_enum(df: pl.DataFrame, schema: dict[str, list[str]]) -> pl.DataFrame:

    return df.with_columns([
        pl.col(c).cast(pl.Enum(vals))
        for c, vals in schema.items()
    ])


def run_pipeline(df: pl.DataFrame) -> pl.DataFrame:

    # Step 1 — split compound column into Department and Region
    df = split_department_region(df)

    # Step 2 — parse Join_Date from M/D/YYYY string format to pl.Date
    df = parse_dates(df, cols=["Join_Date"], fmt="%m/%d/%Y")

    # Step 3 — cast Salary from string to float
    df = cast_numeric(df, {"Salary": pl.Float64})

    # Step 4 — nullify Phone values that are negative (corrupted data)
    df = nullify_invalid(df, "Phone", pl.col("Phone") < 0)

    # Step 5 — fill Age nulls with median (211 nulls found in triage)
    df = fill_nulls_median(df, ["Age"])

    # Step 6 — enforce known category sets for Status and Performance_Score
    df = cast_enum(df, {
        "Status": ["Active", "Inactive", "Pending"],
        "Performance_Score": ["Poor", "Average", "Good", "Excellent"]
    })

    return df


# --- Run the pipeline ---
df_clean = run_pipeline(df)

# Verify results
print(df_clean.schema)       # confirm if all dtypes are correct
print(df_clean.null_count()) # checking for null count

Schema({'Employee_ID': String, 'First_Name': String, 'Last_Name': String, 'Age': Int64, 'Status': Enum(categories=['Active', 'Inactive', 'Pending']), 'Join_Date': Date, 'Salary': Float64, 'Email': String, 'Phone': Int64, 'Performance_Score': Enum(categories=['Poor', 'Average', 'Good', 'Excellent']), 'Remote_Work': Boolean, 'Department': String, 'Region': String})
shape: (1, 13)
┌─────────────┬────────────┬───────────┬─────┬───┬─────────────┬─────────────┬────────────┬────────┐
│ Employee_ID ┆ First_Name ┆ Last_Name ┆ Age ┆ … ┆ Performance ┆ Remote_Work ┆ Department ┆ Region │
│ ---         ┆ ---        ┆ ---       ┆ --- ┆   ┆ _Score      ┆ ---         ┆ ---        ┆ ---    │
│ u32         ┆ u32        ┆ u32       ┆ u32 ┆   ┆ ---         ┆ u32         ┆ u32        ┆ u32    │
│             ┆            ┆           ┆     ┆   ┆ u32         ┆             ┆            ┆        │
╞═════════════╪════════════╪═══════════╪═════╪═══╪═════════════╪═════════════╪════════════╪════════╡
│ 0          